# 🧷 フラグの値域と MTP 検証条件（wheel 静的解析）

`--quantization awq` / `--kv-cache-dtype auto` が通るか、`method=mtp` に何が要求されるか。


In [ ]:
# 使うフラグの「値域」と MTP の検証条件を wheel から確認（install 不要）
import glob
import os
import re
import subprocess
import sys
import zipfile

r = subprocess.run([sys.executable, "-m", "pip", "download", "--no-cache-dir",
                    "--no-deps", "vllm==0.32.0", "-d", "/tmp/w"],
                   capture_output=True, text=True)
whl = glob.glob("/tmp/w/*.whl")
print("download rc:", r.returncode, [os.path.basename(w) for w in whl])
z = zipfile.ZipFile(whl[0])
z.extractall("/tmp/vl")
import pathlib

root = pathlib.Path("/tmp/vl/vllm")

TARGET = ["--host", "--port", "--quantization", "--kv-cache-dtype",
          "--dtype", "--load-format", "--speculative-config",
          "--reasoning-parser", "--max-num-seqs", "--gpu-memory-utilization",
          "--tensor-parallel-size", "--max-model-len", "--served-model-name",
          "--enable-log-requests", "--enforce-eager"]

# add_argument のブロックを雑に集める
hits = {t: [] for t in TARGET}
for f in root.rglob("*.py"):
    try:
        lines = f.read_text(errors="replace").splitlines()
    except Exception:
        continue
    for i, line in enumerate(lines):
        for t in TARGET:
            if f'"{t}"' in line or f"'{t}'" in line:
                ctx = " | ".join(x.strip() for x in lines[i:i + 4])
                hits[t].append(f"{f.relative_to(root)}:{i+1}: {ctx[:220]}")

print("\n=== フラグ定義（値域・choices）===")
for t in TARGET:
    print(f"\n-- {t}: {len(hits[t])} hits")
    for h in hits[t][:4]:
        print("   ", h)

# choices の実値（quantization / kv-cache-dtype / load-format）
for name, rel in (("quantization", "model_executor/layers/quantization/__init__.py"),
                  ("kv", "config/cache.py"),
                  ("load", "config/load.py")):
    f = root / rel
    if not f.exists():
        print("missing:", rel)
        continue
    txt = f.read_text(errors="replace")
    print(f"\n==== {rel} ====")
    for i, line in enumerate(txt.splitlines(), 1):
        s = line.strip()
        if any(k in s for k in ("awq", "auto", "fp8", "CLASS", "Literal",
                                "safetensors", "pt", "dummy", "bitsandbytes")):
            print(f"  {i}: {s[:160]}")

# MTP の検証条件
sp = root / "config/speculative.py"
txt = sp.read_text(errors="replace")
print("\n==== speculative.py: mtp 関連の検証 ====")
for i, line in enumerate(txt.splitlines(), 1):
    s = line.strip()
    if ("mtp" in s.lower() or "raise ValueError" in s
            or "num_speculative_tokens" in s):
        print(f"  {i}: {s[:170]}")
